# **Attribution for transformers: rollout against Chefer**

A practice for the module ["Attribution from axioms"](https://open-xai-platform.web.app).

The lesson makes a claim about attention rollout that can be checked rather than taken on
trust: **the map does not depend on the class** — on a picture with a dog and a cat, rollout
will give the same thing whatever you ask about. And a second one, about Chefer et al.: there
the class specificity is provided by the **gradient of the class logit with respect to the
attention matrix**.

Let us check both. It takes about a minute on a CPU.

**About the implementation, straight away and honestly.** What is assembled below is the
*generic* variant of the Chefer method: the attention matrices are weighted by the gradient, the
positive part is taken, then averaging over heads and a rollout across layers. The full variant
from the paper adds relevance propagation by LRP rules on top of that — with separate rules for
skip connections and for the product $A\cdot V$. It produces cleaner maps but requires
rewriting the whole pass through the network, and does not fit in a notebook. The class
specificity that the whole thing is for comes precisely from the gradient factor, and that is
here.

In [ ]:
!pip install timm -q     # timm is not always installed in Colab

In [ ]:
import io
import types
import urllib.request

import timm
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

torch.manual_seed(0)
DATA = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main'

model = timm.create_model('vit_small_patch16_224', pretrained=True).eval()
for p in model.parameters():
    p.requires_grad_(False)


def attn_forward(self, x, **kw):
    """Attention that keeps the matrix and the gradient with respect to it: both methods need them."""
    B, N, C = x.shape
    qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    q, k = self.q_norm(q), self.k_norm(k)
    attn = ((q * self.scale) @ k.transpose(-2, -1)).softmax(dim=-1)
    self.attn_map = attn          # the matrix is needed by both methods
    attn.retain_grad()            # while the gradient is needed only by Chefer
    return self.proj((attn @ v).transpose(1, 2).reshape(B, N, C))


for block in model.blocks:
    block.attn.fused_attn = False     # fused attention does not hand out the matrix, so we switch it off
    block.attn.forward = types.MethodType(attn_forward, block.attn)

cfg = model.pretrained_cfg
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize(cfg['mean'], cfg['std'])])


def fetch(path):
    return urllib.request.urlopen(f'{DATA}/' + path, timeout=30).read()


img = tf(Image.open(io.BytesIO(fetch('data/cat_and_dog.jpg'))).convert('RGB'))
img = img.unsqueeze(0).requires_grad_(True)
names = [s.strip() for s in fetch('data/imagenet_classes.txt').decode().split('\n')]
GRID = int(model.patch_embed.num_patches ** 0.5)
print(f'blocks: {len(model.blocks)}, patch grid: {GRID}x{GRID}')

## 1. Getting hold of the attention and its gradient

The library computes attention with a fused kernel that does not hand the matrix out. We need
it — both methods do — so fused attention is switched off and a plain four-line implementation
is substituted.

Note `attn.retain_grad()`: without it the gradient with respect to the intermediate tensor is
not kept, and there would be nothing to compute Chefer from.

In [ ]:
def run(cls=None):
    """One pass: the attention matrices of all blocks and the gradients of the class logit."""
    model.zero_grad()
    img.grad = None
    out = model(img)
    c = int(out.argmax()) if cls is None else cls
    out[0, c].backward()
    return c, [b.attn.attn_map.detach()[0] for b in model.blocks], \
        [b.attn.attn_map.grad[0] for b in model.blocks]


predicted, mats, grads = run()
print(f'predicted class: {predicted} {names[predicted]}')
print(f'shape of the attention matrix (heads, tokens, tokens): {tuple(mats[0].shape)}')

## 2. Two methods

The difference between them is one line, and it is worth seeing which one.

**rollout** averages the matrix over heads and adds the identity — that is exactly how skip
connections are accounted for, as the lesson says: half the weight goes to attention, half to
the direct carry-through.

**Chefer** does the same, but before averaging it multiplies the matrix by the gradient of the
class logit and takes the positive part. **The class enters the method exactly here** — nowhere
else in these two functions is it mentioned.

In [ ]:
def rollout(mats):
    """Attention rollout: average over heads, add the identity, multiply across layers."""
    n = mats[0].shape[-1]
    r = torch.eye(n)
    for A in mats:
        a = 0.5 * A.mean(0) + 0.5 * torch.eye(n)     # the identity matrix is exactly how skip connections are accounted for
        r = a @ r
    return r


def chefer(mats, grads):
    """Chefer et al.: the same rollout, but each matrix is weighted by the class-logit gradient."""
    n = mats[0].shape[-1]
    r = torch.eye(n)
    for A, G in zip(mats, grads):
        a = (G * A).clamp(min=0).mean(0)             # this is where the class enters — through the gradient
        a = a + torch.eye(n)
        a = a / a.sum(dim=-1, keepdim=True)
        r = a @ r
    return r


def to_map(r):
    """The row of the CLS token without itself, folded back into the patch grid."""
    m = r[0, 1:].reshape(GRID, GRID)
    return (m - m.min()) / (m.max() - m.min() + 1e-9)

## 3. The main check: does the map depend on the question

We build two maps with each method — asking about the cat and about the dog — and compare.

In [ ]:
CAT, DOG = 281, 243        # tabby and bull mastiff — the cat and the dog
maps = {}
for cls, label in ((CAT, f'cat'), (DOG, f'dog')):
    _, m, g = run(cls)
    maps[label] = (to_map(rollout(m)), to_map(chefer(m, g)))


def corr(a, b):
    a, b = a - a.mean(), b - b.mean()
    return float((a * b).sum() / (a.norm() * b.norm() + 1e-9))


for i, name in ((0, 'rollout'), (1, 'Chefer')):
    a, b = maps[f'cat'][i], maps[f'dog'][i]
    print(f'   {name:9} correlation of the cat and dog maps: {corr(a, b):.6f}   max difference: {(a - b).abs().max():.2e}')

**For rollout the correlation is exactly one and the difference is exactly zero.**
Not "the maps are similar" — they are bit-for-bit identical. The claim of the lesson turned out
not to be a figure of speech: the attention matrix is computed once on the forward pass and
knows nothing about the class, so the product of the matrices knows nothing either.

**For Chefer the correlation is about $0.7$**, and the difference is more than half the range of
the map. The maps are different, and one factor made them so.

**Task 1.** Take a third class — something deliberately unsuitable, say `849` (a teapot). Will
the rollout map stay the same? And how far will the Chefer map be from the one for the dog?

In [ ]:
# Your code here

## 4. Where exactly the maps look

Correlation says the maps are different but does not say whether they are **correctly**
different. Let us mark the boxes of the cat and the dog and see what share of the map mass falls
into each.

In [ ]:
# The boxes are drawn by eye on the 224 by 224 picture
BOXES = {f'cat': (10, 40, 110, 200), f'dog': (115, 30, 215, 205)}


def mass_in(m, box):
    """The share of the map mass falling inside the box."""
    big = F.interpolate(m[None, None], (224, 224), mode='bilinear', align_corners=False)[0, 0]
    x0, y0, x1, y1 = box
    return float(big[y0:y1, x0:x1].sum() / big.sum())


print(f'{"":10}{"asked about":12}{"in the cat box":>16}{"in the dog box":>17}')
for label in (f'cat', f'dog'):
    for i, name in ((0, 'rollout'), (1, 'Chefer')):
        m = maps[label][i]
        print(f'{name:10}{label:12}{mass_in(m, BOXES[f"cat"]) * 100:15.1f}%'
              f'{mass_in(m, BOXES[f"dog"]) * 100:16.1f}%')

This table is what the whole thing was for.

**The rollout rows coincide to a tenth of a percent** — not because the method is bad, but
because it answers the question "where does the model look at all", not "why this particular
class". Just like Eigen-CAM from the module on the CAM family.

**The Chefer rows differ, and differ in the right direction:** ask about the cat and more mass
lands in the cat box; ask about the dog and the emphasis moves to the dog box. That is class
specificity, measured with a number.

**Task 2.** Build the same table for a class that is not in the picture at all. Where does the
mass land for Chefer? What does that say: has the method found a non-existent object, or has it
honestly shown what the model would latch onto if it were forced to answer about that class?

In [ ]:
# Your code here

## 5. Looking with the eyes

The numbers have said the main thing, but the maps are worth seeing too.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for row, label in enumerate((f'cat', f'dog')):
    for col, name in enumerate(('rollout', 'Chefer')):
        axes[row, col].imshow(maps[label][col].numpy(), cmap='inferno')
        axes[row, col].set_title(f'{name} — {label}')
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()

**Task 3.** The lesson names a third option for transformers — Grad-CAM on
reshaped tokens, from the module on the CAM family. Build it on the same picture and for the
same two classes and add a row to the box table. Where will it land between rollout and Chefer —
closer to the class-blind or to the class-specific one?

In [ ]:
# Your code here

## What to take away from this notebook

- **Attention rollout is class-blind, and this is verified bit for bit.** The correlation of the
  maps for two different classes equals one, the difference equals zero. If you need an answer
  to "why this particular class", rollout will never give it, however much you improve the
  implementation.
- **The class enters Chefer exactly through the gradient**, and that is visible in the code: one
  line in which the attention matrix is multiplied by $\partial y^c/\partial A$. Remove it and
  you get rollout.
- **Correlation is not enough, a box is needed.** "The maps are different" is not yet "the maps
  are right". The share of mass inside an annotated region answers the second question, and
  answers it with a number.
- **The full Chefer is more complex than what is assembled here**: it adds LRP rules for skip
  connections and for the product $A\cdot V$. The maps come out cleaner, but the class
  specificity comes not from those rules but from the gradient factor — the one already here.